# YouTube AI-Slop: transcript-routed synthetic-voice experiment

This Colab notebook tests whether an AI-writing detector can **route** expensive synthetic-voice inference toward useful parts of a video.

It does not treat AI-written text as proof of AI voice. It also samples uniform control clips, because synthetic speech can read human-written text and people can read AI-written scripts.

> **Runtime:** select **Runtime → Change runtime type → T4 GPU** before running.  
> **License:** DF Arena 1B has a non-commercial model license. Use this notebook for evaluation/prototyping only unless you separately resolve model licensing.

## Goal

Produce inspectable, clip-level evidence:

- timestamped transcript;
- AI-writing scores over overlapping 80–150-word regions;
- 4.04-second synthetic-voice scores from text-ranked and uniform-control clips; and
- conservative video-level aggregates for comparing experiments.

The output is not a production block decision.

## Setup

Install the Colab dependencies. PyTorch and FFmpeg are provided by Colab.

In [ ]:
from pathlib import Path
import subprocess
import sys

REQUIREMENTS_URL = "https://raw.githubusercontent.com/emilzmmn04/youtube-ai-slop-audio-detector/main/requirements-colab.txt"
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", REQUIREMENTS_URL])
print("Dependencies installed. If Colab asks for a runtime restart, restart and continue here.")

## Configuration

Start with the defaults. Raising sample counts improves coverage but increases GPU time.

In [ ]:
# Input: paste a URL, or leave blank to upload a local audio/video file.
YOUTUBE_URL = ""  # @param {type:"string"}

# Models
ASR_MODEL_ID = "openai/whisper-small"
TEXT_MODEL_ID = "GeorgeDrayson/modernbert-ai-detection-raid-mage"
AUDIO_MODEL_ID = "Speech-Arena-2025/DF_Arena_1B_V_1"

# Transcript routing
TEXT_WINDOW_WORDS = 120  # @param {type:"integer"}
TEXT_STRIDE_WORDS = 60  # @param {type:"integer"}
TOP_TEXT_WINDOWS = 3  # @param {type:"integer"}
CLIPS_PER_TEXT_WINDOW = 3  # @param {type:"integer"}
UNIFORM_CONTROL_CLIPS = 6  # @param {type:"integer"}

# DF Arena's own feature extractor uses 64,600 samples at 16 kHz (~4.04 s).
SAMPLE_RATE = 16_000
AUDIO_CLIP_SAMPLES = 64_600
AUDIO_CLIP_SECONDS = AUDIO_CLIP_SAMPLES / SAMPLE_RATE
PROVISIONAL_AUDIO_THRESHOLD = 0.80  # @param {type:"number"}

assert 40 <= TEXT_WINDOW_WORDS <= 500
assert 1 <= TEXT_STRIDE_WORDS <= TEXT_WINDOW_WORDS
assert TOP_TEXT_WINDOWS >= 1
print(f"Audio clips: {AUDIO_CLIP_SECONDS:.4f} seconds each")

## Step 1 — Acquire and normalize media

The notebook first tries `yt-dlp`. If `YOUTUBE_URL` is empty, Colab opens a file uploader. All input is converted to 16 kHz mono WAV. YouTube sometimes blocks datacenter downloads; upload the file manually if that occurs.

In [ ]:
import shutil
from pathlib import Path

WORKDIR = Path("/content/ai_slop_detector")
WORKDIR.mkdir(parents=True, exist_ok=True)
SOURCE_PATH = WORKDIR / "input_media"
AUDIO_PATH = WORKDIR / "input_audio.wav"

if YOUTUBE_URL.strip():
    output_template = str(WORKDIR / "download.%(ext)s")
    command = [
        "yt-dlp", "--no-playlist", "-f", "bestaudio/best",
        "-o", output_template, YOUTUBE_URL.strip(),
    ]
    subprocess.run(command, check=True)
    downloaded = [p for p in WORKDIR.glob("download.*") if p.is_file()]
    if not downloaded:
        raise FileNotFoundError("yt-dlp finished but no media file was found")
    source_media = downloaded[0]
else:
    try:
        from google.colab import files
    except ImportError as exc:
        raise RuntimeError("File upload requires Colab; alternatively set YOUTUBE_URL") from exc
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError("No file uploaded")
    uploaded_name = next(iter(uploaded))
    source_media = WORKDIR / Path(uploaded_name).name
    source_media.write_bytes(uploaded[uploaded_name])

subprocess.run([
    "ffmpeg", "-y", "-i", str(source_media), "-vn",
    "-ac", "1", "-ar", str(SAMPLE_RATE), "-c:a", "pcm_s16le", str(AUDIO_PATH),
], check=True, capture_output=True)

print(f"Normalized audio: {AUDIO_PATH}")
print(f"Size: {AUDIO_PATH.stat().st_size / 1_000_000:.1f} MB")

## Step 2 — Transcribe with word timestamps

Whisper creates the transcript and the time mapping used to return from text windows to audio. The ASR model is deleted afterward to leave GPU memory for DF Arena.

In [ ]:
import gc
import json
import torch
from transformers import pipeline

if not torch.cuda.is_available():
    raise RuntimeError("A GPU runtime is strongly recommended. In Colab choose Runtime → Change runtime type → T4 GPU.")

asr = pipeline(
    "automatic-speech-recognition",
    model=ASR_MODEL_ID,
    device=0,
    torch_dtype=torch.float16,
    chunk_length_s=30,
    batch_size=8,
)
transcription = asr(
    str(AUDIO_PATH),
    return_timestamps="word",
    generate_kwargs={"task": "transcribe"},
)

words = []
for chunk in transcription.get("chunks", []):
    start_s, end_s = chunk.get("timestamp", (None, None))
    token = chunk.get("text", "").strip()
    if token and start_s is not None and end_s is not None:
        words.append({"text": token, "start_s": float(start_s), "end_s": float(end_s)})

if not words:
    raise RuntimeError("No timestamped speech was found")

transcript_record = {"text": transcription["text"], "words": words}
(WORKDIR / "transcript.json").write_text(json.dumps(transcript_record, indent=2), encoding="utf-8")
print(f"Transcribed {len(words):,} timestamped words ({words[-1]['end_s'] / 60:.1f} min).")
print(transcription["text"][:800])

del asr
gc.collect()
torch.cuda.empty_cache()

## Step 3 — Build and score transcript windows

AI-writing classifiers are unreliable on tiny snippets, so the routing stage scores overlapping long windows. The text score ranks regions; it is not treated as a factual probability or blocking signal.

In [ ]:
import numpy as np
import pandas as pd
from transformers import AutoModelForSequenceClassification, AutoTokenizer


def build_text_windows(word_records, window_words, stride_words):
    rows = []
    total = len(word_records)
    starts = list(range(0, max(total - window_words + 1, 1), stride_words))
    final_start = max(total - window_words, 0)
    if not starts or starts[-1] != final_start:
        starts.append(final_start)
    for window_id, start_index in enumerate(sorted(set(starts))):
        end_index = min(start_index + window_words, total)
        window = word_records[start_index:end_index]
        rows.append({
            "window_id": window_id,
            "start_word": start_index,
            "end_word": end_index,
            "start_s": window[0]["start_s"],
            "end_s": window[-1]["end_s"],
            "word_count": len(window),
            "text": " ".join(item["text"] for item in window),
        })
    return pd.DataFrame(rows)


text_windows = build_text_windows(words, TEXT_WINDOW_WORDS, TEXT_STRIDE_WORDS)
tokenizer = AutoTokenizer.from_pretrained(TEXT_MODEL_ID)
text_model = AutoModelForSequenceClassification.from_pretrained(TEXT_MODEL_ID).to("cuda").eval()

scores = []
batch_size = 8
for offset in range(0, len(text_windows), batch_size):
    batch_text = text_windows.iloc[offset:offset + batch_size]["text"].tolist()
    inputs = tokenizer(batch_text, padding=True, truncation=True, max_length=1024, return_tensors="pt")
    inputs = {name: value.to("cuda") for name, value in inputs.items()}
    with torch.inference_mode():
        logits = text_model(**inputs).logits.float()
    if logits.shape[-1] == 1:
        batch_scores = torch.sigmoid(logits[:, 0])
    else:
        # The model's research usage defines class index 1 as machine-generated.
        batch_scores = torch.softmax(logits, dim=-1)[:, 1]
    scores.extend(batch_scores.cpu().tolist())

text_windows["ai_text_score"] = scores
text_windows["text_rank"] = text_windows["ai_text_score"].rank(method="first", ascending=False).astype(int)
text_windows.sort_values("text_rank").to_csv(WORKDIR / "transcript_windows.csv", index=False)
display(text_windows.sort_values("text_rank").head(10)[
    ["text_rank", "start_s", "end_s", "word_count", "ai_text_score", "text"]
])

del text_model, tokenizer, inputs, logits
gc.collect()
torch.cuda.empty_cache()

## Step 4 — Choose text-ranked and control audio clips

Each candidate is exactly the duration consumed by DF Arena. Uniform controls preserve coverage outside AI-looking transcript regions. Near-duplicate timestamps are removed.

In [ ]:
def evenly_spaced(values, count):
    if not values or count <= 0:
        return []
    indexes = np.linspace(0, len(values) - 1, min(count, len(values))).round().astype(int)
    return [values[index] for index in sorted(set(indexes.tolist()))]


def nearest_text_score(center_s):
    overlapping = text_windows[(text_windows.start_s <= center_s) & (text_windows.end_s >= center_s)]
    if len(overlapping):
        return float(overlapping.ai_text_score.max())
    nearest_index = ((text_windows.start_s + text_windows.end_s) / 2 - center_s).abs().idxmin()
    return float(text_windows.loc[nearest_index, "ai_text_score"])


candidates = []
top_windows = text_windows.nsmallest(TOP_TEXT_WINDOWS, "text_rank")
for row in top_windows.itertuples():
    region_words = words[row.start_word:row.end_word]
    centers = [(item["start_s"] + item["end_s"]) / 2 for item in region_words]
    for center_s in evenly_spaced(centers, CLIPS_PER_TEXT_WINDOW):
        candidates.append({
            "source": "text_ranked",
            "center_s": center_s,
            "ai_text_score": float(row.ai_text_score),
            "text_rank": int(row.text_rank),
        })

all_speech_centers = [(item["start_s"] + item["end_s"]) / 2 for item in words]
for center_s in evenly_spaced(all_speech_centers, UNIFORM_CONTROL_CLIPS):
    candidates.append({
        "source": "uniform_control",
        "center_s": center_s,
        "ai_text_score": nearest_text_score(center_s),
        "text_rank": None,
    })

deduplicated = []
for candidate in sorted(candidates, key=lambda item: (item["center_s"], item["source"])):
    if any(abs(candidate["center_s"] - prior["center_s"]) < AUDIO_CLIP_SECONDS / 2 for prior in deduplicated):
        continue
    start_s = max(0.0, candidate["center_s"] - AUDIO_CLIP_SECONDS / 2)
    candidate["start_s"] = start_s
    candidate["end_s"] = start_s + AUDIO_CLIP_SECONDS
    deduplicated.append(candidate)

candidate_clips = pd.DataFrame(deduplicated)
display(candidate_clips)
print(candidate_clips["source"].value_counts())

## Step 5 — Score clips with DF Arena 1B

The repository's custom model accepts one raw-waveform clip at a time. Short tail clips are repeated to the model's required 64,600 samples, matching its published feature extractor. `spoof_score` is the model's softmax score for synthetic speech, not a calibrated probability.

In [ ]:
import librosa
from transformers import AutoModel

waveform, loaded_rate = librosa.load(AUDIO_PATH, sr=SAMPLE_RATE, mono=True)
assert loaded_rate == SAMPLE_RATE

audio_dtype = torch.float16
audio_model = AutoModel.from_pretrained(
    AUDIO_MODEL_ID,
    trust_remote_code=True,
    torch_dtype=audio_dtype,
    low_cpu_mem_usage=True,
).to("cuda").eval()
spoof_index = int(audio_model.config.label2id["spoof"])


def extract_fixed_clip(start_s):
    start_sample = max(0, int(round(start_s * SAMPLE_RATE)))
    clip = waveform[start_sample:start_sample + AUDIO_CLIP_SAMPLES]
    if len(clip) == 0:
        raise ValueError(f"No audio available at {start_s:.2f}s")
    if len(clip) < AUDIO_CLIP_SAMPLES:
        repeats = int(np.ceil(AUDIO_CLIP_SAMPLES / len(clip)))
        clip = np.tile(clip, repeats)[:AUDIO_CLIP_SAMPLES]
    return clip.astype(np.float32, copy=False)


spoof_scores = []
predicted_labels = []
for row in candidate_clips.itertuples():
    clip = extract_fixed_clip(row.start_s)
    input_values = torch.from_numpy(clip).to(device="cuda", dtype=audio_dtype)
    with torch.inference_mode():
        output = audio_model(input_values=input_values)
        probabilities = torch.softmax(output["logits"].float(), dim=-1)[0]
    spoof_score = float(probabilities[spoof_index].cpu())
    spoof_scores.append(spoof_score)
    predicted_labels.append("spoof" if spoof_score >= 0.5 else "bonafide")

candidate_clips["spoof_score"] = spoof_scores
candidate_clips["model_label"] = predicted_labels
candidate_clips["above_provisional_threshold"] = (
    candidate_clips["spoof_score"] >= PROVISIONAL_AUDIO_THRESHOLD
)
candidate_clips.sort_values("spoof_score", ascending=False).to_csv(WORKDIR / "clip_scores.csv", index=False)

display(candidate_clips.sort_values("spoof_score", ascending=False))

del audio_model, waveform
gc.collect()
torch.cuda.empty_cache()

## Checks — Aggregate without overclaiming

The median of the three highest audio scores is less sensitive to one anomalous clip than the maximum. Requiring multiple positive clips is a provisional experiment rule, not a calibrated YouTube blocker.

In [ ]:
sorted_audio_scores = sorted(candidate_clips["spoof_score"].tolist(), reverse=True)
top_count = min(3, len(sorted_audio_scores))
audio_peak_median = float(np.median(sorted_audio_scores[:top_count]))
evidence_count = int(candidate_clips["above_provisional_threshold"].sum())

summary = {
    "duration_minutes": round(words[-1]["end_s"] / 60, 3),
    "timestamped_words": len(words),
    "text_windows": len(text_windows),
    "clips_tested": len(candidate_clips),
    "text_ranked_clips": int((candidate_clips.source == "text_ranked").sum()),
    "uniform_control_clips": int((candidate_clips.source == "uniform_control").sum()),
    "max_ai_text_score": float(text_windows.ai_text_score.max()),
    "max_spoof_score": float(candidate_clips.spoof_score.max()),
    "median_top_3_spoof_score": audio_peak_median,
    "provisional_audio_threshold": PROVISIONAL_AUDIO_THRESHOLD,
    "clips_above_provisional_threshold": evidence_count,
    "provisional_multi_clip_evidence": evidence_count >= 2,
    "warning": "Experimental, uncalibrated evidence. Do not use as an automatic block decision.",
}
(WORKDIR / "run_summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
display(pd.Series(summary, name="value").to_frame())

## Inspect and export

Listen to the highest-scoring clips before interpreting the result. Music, overlapping speakers, silence, compression, and ASR errors are useful failure cases to record.

In [ ]:
from IPython.display import Audio, display

# Reload only for listening; this does not run either model.
listen_waveform, _ = librosa.load(AUDIO_PATH, sr=SAMPLE_RATE, mono=True)
for row in candidate_clips.nlargest(min(5, len(candidate_clips)), "spoof_score").itertuples():
    start = int(row.start_s * SAMPLE_RATE)
    end = start + AUDIO_CLIP_SAMPLES
    print(f"{row.start_s:.2f}s | {row.source} | spoof={row.spoof_score:.3f} | text={row.ai_text_score:.3f}")
    display(Audio(listen_waveform[start:end], rate=SAMPLE_RATE))

print("Artifacts:")
for artifact in ["transcript.json", "transcript_windows.csv", "clip_scores.csv", "run_summary.json"]:
    print(WORKDIR / artifact)

# Optional download:
# from google.colab import files
# for artifact in ["transcript.json", "transcript_windows.csv", "clip_scores.csv", "run_summary.json"]:
#     files.download(str(WORKDIR / artifact))

## Next steps

Run the same labeled videos under three policies:

1. uniform audio clips only;
2. text-ranked clips only; and
3. text-ranked plus uniform controls (this notebook).

Evaluate at the **video level** with complete videos/channels held out. Record precision, recall, AUROC, and false blocks per 100 hours. Repeat after YouTube-like AAC/Opus re-encoding. Only after collecting those results should the transcript and audio evidence be calibrated into a combined classifier.